# First Try... not very successful

In [1]:
import tkinter as tk
from tkinter import filedialog, ttk, messagebox
import h5py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import openmatrix as omx
from datetime import datetime, timedelta

In [2]:
file_path = '/Users/minhaohuang/Desktop/Grad Assistant/Code_to_h51.h5'

with h5py.File(file_path, 'r') as h5_file:
    print("Keys (top-level group):")
    h5_file.visit(lambda name: print(name))

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/Users/minhaohuang/Desktop/Grad Assistant/Code_to_h51.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
def explore_h5(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"Dataset: {name}, shape: {obj.shape}, dtype: {obj.dtype}")
    elif isinstance(obj, h5py.Group):
        print(f"Group: {name}")

with h5py.File(file_path, 'r') as h5_file:
    h5_file.visititems(explore_h5)


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/Users/minhaohuang/Desktop/Grad Assistant/Code_to_h51.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
with h5py.File(file_path, 'r') as h5_file:
    data = h5_file['DS1'][:10]  # Get first 10 values
    print(data)


[[ 7.37404581e+05  7.37404581e+05  7.37404581e+05 ...  7.37407580e+05
   7.37407580e+05  7.37407580e+05]
 [-1.60156250e-01  1.65463034e-02  1.74187257e-01 ... -5.75470254e-02
  -5.56836039e-02 -5.85937500e-02]
 [-6.66015625e-01 -6.87143104e-01 -6.90346430e-01 ... -7.51238244e-01
  -7.52202151e-01 -7.52693557e-01]
 ...
 [-8.47000000e+00 -9.01331025e+00 -8.71360691e+00 ... -9.10000000e-01
  -9.69486783e-01 -1.20808995e+00]
 [-2.66000000e+00 -2.59333795e+00 -2.66672138e+00 ...  1.40000000e-01
   9.53849126e-02 -1.57301578e-02]
 [ 1.40000000e+00  1.34001385e+00  1.54672138e+00 ... -1.10843140e+00
  -1.09461509e+00 -1.22382011e+00]]


In [ ]:
###Creates H5 data window

def matlab_to_datetime(datenum):
    return datetime.fromordinal(int(datenum)) + timedelta(days=datenum % 1) - timedelta(days=366)
    
class H5Viewer:
    def __init__(self, root):
        self.root = root
        self.root.title("HDF5 Time Selector")
        
        self.load_button = tk.Button(root, text="Load .h5 File", command=self.load_file)
        self.load_button.pack(pady=10)

        self.time_label = tk.Label(root, text="Select Time:")
        self.time_label.pack()

        self.time_combo = ttk.Combobox(root, state="readonly")
        self.time_combo.pack()

        self.view_button = tk.Button(root, text="View Data", command=self.view_data)
        self.view_button.pack(pady=10)

        self.h5_file = None
        self.time_data = None

    def load_file(self):
        filepath = filedialog.askopenfilename(filetypes=[("HDF5 files", "*.h5")])
        if not filepath:
            return
        
        self.h5_file = h5py.File(filepath, 'r')
        print("Loaded file:", filepath)

        # Try accessing time dataset (customize this to your file structure)
        if 'time' in self.h5_file:
            self.time_data = self.h5_file[0][:]
        else:
            tk.messagebox.showerror("Error", "No 'time' dataset found in the file.")
            return

        time_strs = [str(t) for t in self.time_data]
        self.time_combo['values'] = time_strs
        if time_strs:
            self.time_combo.current(0)

    def view_data(self):
        selected_time = self.time_combo.get()
        if not selected_time:
            tk.messagebox.showwarning("Warning", "No time selected.")
            return

        # Dummy example: get data based on index (customize this to your file layout)
        time_index = list(self.time_data).index(float(selected_time))
        data = self.h5_file['your_data'][time_index]  # Replace 'your_data' accordingly

        plt.figure()
        plt.plot(data)  # Or whatever visualization makes sense
        plt.title(f"Data at time {selected_time}")
        plt.show()

root = tk.Tk()
app = H5Viewer(root)
root.mainloop()


Loaded file: C:/Users/ahala/OneDrive/Desktop/SoangraResearch/McRobertsAnalysis/Data/H5_Files/TestData.h5


Frequency is 100Hz when we need to truncate the data

In [ ]:
# --- Helper: Convert MATLAB datenum to Python datetime ---
def matlab_datenum_to_datetime(matlab_datenum):
    return datetime.fromordinal(int(matlab_datenum)) + \
           timedelta(days=matlab_datenum % 1) - timedelta(days=366)

# --- Read .h5 file ---
file_path = '/Users/minhaohuang/Desktop/Grad Assistant/Code_to_h51.h5'

with h5py.File(file_path, 'r') as f:
    # Show dataset keys
    print("Datasets:")
    for key in f.keys():
        print(f" - {key}")

    # Load mtime from first column of 'DS1'
    mtime_array = np.array(f['DS1'][:, 0])  # shape (N,)

# --- Filter: keep only valid MATLAB datenum values ---
valid_mask = (mtime_array > 700000) & (mtime_array < 800000)
valid_mtimes = mtime_array[valid_mask]

# --- Convert to Python datetime ---
dates = [matlab_datenum_to_datetime(ts) for ts in valid_mtimes]

# --- Print the first few converted dates ---
print("\nFirst few converted timestamps:")
for i, d in enumerate(dates[:5]):
    print(f"{i}: {d}")


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/Users/minhaohuang/Desktop/Grad Assistant/Code_to_h51.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
print("Min mtime:", mtime_array.min())
print("Bad values:", mtime_array[mtime_array < 1])
print("Max mtime:", mtime_array.max())

Min mtime: -8.47
Bad values: [-0.16015625 -0.66601562 -0.48242188 -8.47       -2.66      ]
Max mtime: 737404.5805406712


In [ ]:
# --- Helper: MATLAB datenum to datetime ---
def matlab_datenum_to_datetime(matlab_datenum):
    return datetime.fromordinal(int(matlab_datenum)) + \
           timedelta(days=matlab_datenum % 1) - timedelta(days=366)

# --- File paths ---
input_path = '/Users/minhaohuang/Desktop/Grad Assistant/Code_to_h51.h5'
output_path = '/Users/minhaohuang/Desktop/Grad Assistant/Code_converted1.h5'

# --- Load data ---
with h5py.File(input_path, 'r') as f:
    data = np.array(f['DS1'])  # shape: (num_features, num_samples)
    mtime_row = data[0, :]     # First row = MATLAB datenum for each sample

# --- Filter valid datenum range ---
valid_mask = (mtime_row > 700000) & (mtime_row < 800000)
safe_mtime = mtime_row.copy()
safe_mtime[~valid_mask] = np.nan  # Mark invalids

# --- Convert to datetime strings ---
datetime_strings = np.array([
    matlab_datenum_to_datetime(ts).strftime('%Y-%m-%d %H:%M:%S') if not np.isnan(ts) else ''
    for ts in safe_mtime
])

# --- Convert to fixed-length byte strings for HDF5 ---
datetime_bytes = np.array(datetime_strings, dtype='S19')  # 'YYYY-MM-DD HH:MM:SS'

# --- Save to new HDF5 file ---
with h5py.File(output_path, 'w') as f_out:
    f_out.create_dataset('DS1_original', data=data)
    f_out.create_dataset('DS1_datetime_str', data=datetime_bytes)

print("✅ Saved to:", output_path)
print("📌 Shape of DS1:", data.shape)
print("🕒 First datetime sample:", datetime_strings[0])


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/Users/minhaohuang/Desktop/Grad Assistant/Code_to_h51.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
with h5py.File(output_path, 'r') as h5_file:
    print("Keys (top-level group):")
    h5_file.visit(lambda name: print(name))

Keys (top-level group):
DS1_datetime_str


In [ ]:
def explore_h5(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"Dataset: {name}, shape: {obj.shape}, dtype: {obj.dtype}")
    elif isinstance(obj, h5py.Group):
        print(f"Group: {name}")

with h5py.File(output_path, 'r') as h5_file:
    h5_file.visititems(explore_h5)


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/Users/minhaohuang/Desktop/Grad Assistant/Code_converted1.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
with h5py.File(output_path, 'r') as h5_file:
    data = h5_file['DS1_datetime_str'][:50] 
    print(data)


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/Users/minhaohuang/Desktop/Grad Assistant/Code_converted1.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

# Second Try

In [ ]:
# from oct2py import Oct2Py
# import tkinter as tk

# root = tk.Tk()
# export_format_var = tk.StringVar(value="csv")
# oc = Oct2Py()
# oc.addpath('/Users/minhaohuang/Desktop/Grad Assistant/Code_to_h5/DynaportCompany_Codes_new')

# # # Add export format dropdown
# # tk.Label(root, text="Select Export Format:").pack()
# # format_menu = ttk.Combobox(root, textvariable=export_format_var, values=["csv", "h5", "json"], state="readonly")
# # format_menu.pack()


In [8]:
import sys
import os

base_path = "C:/Users/ahala/OneDrive/Desktop/SoangraResearch/McRobertsAnalysis"  ##Replace with base path

if base_path not in sys.path:
    sys.path.append(base_path)

import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import pandas as pd
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from GSDA_code import gsd_iluz # Mobilise-D (healthy)
from GSDB_code import resamp_interp
from GSDB_code import gsd_low_back_acc # Mobilise-D (patients)
from ICDA_code import step_detection
from Non_GUI.Ayush_Files.gait_functions.walking import find_walking # Ayush's function
from Non_GUI.Ayush_Files.gait_functions.hs_to_detection import get_hs_to_wavelet # Ayush's function
from Non_GUI.Ayush_Files.gait_functions.hs_to_detection import gait_parameters # Ayush's function
from Non_GUI.Ayush_Files.gait_functions.hs_to_detection import get_wavelet
from Non_GUI.Ayush_Files.gait_functions.hs_to_detection import get_scales

# Create the main window
root = tk.Tk()
root.title("Export Tool")
root.geometry("1000x600")

tab_control = ttk.Notebook(root)
tab1 = tk.Frame(tab_control)
tab2 = tk.Frame(tab_control)
tab3 = tk.Frame(tab_control)
tab_control.add(tab1, text="Extract Data") #First Tab
tab_control.add(tab2, text="Mobilise-D") #Second Tab
tab_control.add(tab3, text="Ayush's Code") #Third Tab
tab_control.pack(expand=1, fill='both')


# Global variables
# selected_file = None
# extracted_df = None
start_time_var = tk.StringVar()
stop_time_var = tk.StringVar()
file_start_time_var = tk.StringVar()
file_stop_time_var = tk.StringVar()
label_var = tk.StringVar()
export_format_var = tk.StringVar()

# -------------------------Tab 1 Extract Data---------------------------------
# Load file function
def load_file():
    global selected_file #, extracted_df
    file_path = filedialog.askopenfilename(filetypes=[("H5 files", "*.h5")])
    if not file_path:
        return

    selected_file = file_path
    file_label.config(text=f"Loaded: {os.path.basename(file_path)}")

    try:
        with h5py.File(selected_file, "r") as f:
            dataset = list(f.values())[0][()]
            time_row = dataset[0, :]

            def matlab_datenum_to_datetime(datenum):
                days = int(datenum)
                frac = datenum % 1
                return datetime.fromordinal(days - 366) + timedelta(days=frac)

            time_array = [matlab_datenum_to_datetime(t) for t in time_row]

            file_start_time = time_array[0].strftime("%Y-%m-%d %H:%M:%S")
            file_stop_time = time_array[-1].strftime("%Y-%m-%d %H:%M:%S")

            # Update non-editable display
            file_start_time_var.set(file_start_time)
            file_stop_time_var.set(file_stop_time)

            # Set defaults for editable fields
            start_time_var.set(file_start_time)
            stop_time_var.set(file_stop_time)

        messagebox.showinfo("File Loaded", f"File loaded successfully:\n{file_path}")
    except Exception as e:
        messagebox.showerror("Error", f"Failed to read file:\n{str(e)}")
        
load_button1 = tk.Button(tab1, text="Load H5 File", command=load_file)
load_button1.pack(pady=10)

file_label = tk.Label(tab1, text="No file loaded")
file_label.pack()

tk.Label(tab1, text="File Start Time:").pack()
file_start_entry = tk.Entry(tab1, textvariable=file_start_time_var, state='readonly')
file_start_entry.pack()

tk.Label(tab1, text="File Stop Time:").pack()
file_stop_entry = tk.Entry(tab1, textvariable=file_stop_time_var, state='readonly')
file_stop_entry.pack()

def save_dataframe_as_csv(df):
    output_path = filedialog.asksaveasfilename(
        defaultextension=".csv",
        filetypes=[("CSV files", "*.csv")],
        # initialfile="run1.csv",
        title="Save CSV File"
    )

    if not output_path:
        return False  # User cancelled the save dialog

    try:
        df.to_csv(output_path, index=False)
        messagebox.showinfo("Exported", f"Data exported to:\n{output_path}")
        return True
    except Exception as e:
        messagebox.showerror("Error", f"Failed to save file:\n{str(e)}")
        return False

def export_data():
    if not selected_file:
        messagebox.showerror("No File", "Please load a file first.")
        return

    try:
        start_time = datetime.strptime(start_time_var.get(), "%Y-%m-%d %H:%M:%S")
        stop_time = datetime.strptime(stop_time_var.get(), "%Y-%m-%d %H:%M:%S")
    except ValueError:
        messagebox.showerror("Invalid Time", "Time must be in format: YYYY-MM-DD HH:MM:SS")
        return

    try:
        with h5py.File(selected_file, "r") as f:
            # Assuming the data is stored in the first dataset
            dataset = list(f.values())[0][()]
        
        # First row is time in MATLAB datenum format
        time_row = dataset[0, :]
        rest_data = dataset[1:, :]

        # Convert datenum to datetime using manual conversion
        def matlab_datenum_to_datetime(datenum):
            days = int(datenum)
            frac = datenum % 1
            return datetime.fromordinal(days - 366) + timedelta(days=frac)

        time_array = [matlab_datenum_to_datetime(t) for t in time_row]

        # Find indices within start and stop time
        indices = [i for i, t in enumerate(time_array) if start_time <= t <= stop_time]

        if not indices:
            messagebox.showerror("No Data", "No data found in the selected time range.")
            return

        # Extract subset
        extracted = pd.DataFrame(dataset[:, indices]).iloc[0:4]
        df = extracted.T
        df.columns = ['Time','Ax','Ay','Az']

        save_dataframe_as_csv(df)
    except Exception as e:
        messagebox.showerror("Error", f"Failed to export data:\\n{str(e)}")

tk.Label(tab1, text="Start Time (YYYY-MM-DD HH:MM:SS):").pack()
start_entry = tk.Entry(tab1, textvariable=start_time_var)
start_entry.pack()

tk.Label(tab1, text="Stop Time (YYYY-MM-DD HH:MM:SS):").pack()
stop_entry = tk.Entry(tab1, textvariable=stop_time_var)
stop_entry.pack()

format_frame = tk.Frame(tab1)
format_frame.pack(pady=5)
for fmt in ["CSV", "JSON"]:
    tk.Radiobutton(format_frame, text=fmt.upper(), variable=export_format_var, value=fmt).pack(side='left', padx=15)

export_button = tk.Button(tab1, text="Export", command=export_data)
export_button.pack(pady=20)

# --------------------------Tab 2 Mobilise D----------------------------------
def load_csv_json():
    global extracted_df
    file_path = filedialog.askopenfilename(filetypes=[("CSV/JSON files", "*.csv *.json")])
    if not file_path:
        return

    try:
        if file_path.endswith(".csv"):
            df = pd.read_csv(file_path)
        elif file_path.endswith(".json"):
            df = pd.read_json(file_path)
        else:
            raise ValueError("Unsupported file format")

        extracted_df = df.copy()
        messagebox.showinfo("Loaded", f"Loaded file:\n{file_path}")
    except Exception as e:
        messagebox.showerror("Load Error", str(e))

tk.Button(tab2, text="Upload CSV/JSON File", command=load_csv_json).pack(pady=10)

tk.Label(tab2, text="Select Type").pack(pady=(20, 5))
label_frame = tk.Frame(tab2)
label_frame.pack()
for lbl in ["healthy(MD)", "patient(MD)"]:
    tk.Radiobutton(label_frame, text=lbl.capitalize(), variable=label_var, value=lbl).pack(side="left", padx=20)

def process_gsd():
    global extracted_df, sd_result_df, step_result_df
    if extracted_df is None:
        messagebox.showerror("Error", "No extracted data.")
        return
    def matlab_datenum_to_datetime(datenum):
        days = int(datenum)
        frac = datenum % 1
        return datetime.fromordinal(days - 366) + timedelta(days=frac)
    try:
        acc = extracted_df.iloc[:, 1:4].to_numpy()
        time_col = extracted_df['Time'].to_numpy()
        original_time_array = [matlab_datenum_to_datetime(t) for t in time_col]
        fs = 100 

        if label_var.get() == "healthy(MD)":
            from GSDA_code import gsd_iluz
            imu_input = {"LowerBack": {"Acc": acc, "Gyr": np.zeros_like(acc), "Fs": {"Acc": fs}}}
            gsd_result = gsd_iluz(imu_input, time_vector=original_time_array, plot_results=True)
        else: #label_var.get() == "patient(MD)":
            from GSDB_code import gsd_low_back_acc
            gsd_result = gsd_low_back_acc(acc, fs, plot_results=True, time_vector=original_time_array)
        
        messagebox.showinfo("GSD Result", f"Gait sequences found: {len(gsd_result)}")
    except Exception as e:
        messagebox.showerror("Error", str(e))

    sd_result_df = pd.DataFrame(gsd_result)
    step_result = step_detection(acc, gsd_result, fs, plot_results=False)
    step_result_df = pd.DataFrame(step_result)
    
tk.Button(tab2, text="Run GSD Detection", command=process_gsd).pack(pady=15)

def save_gsd_result():
    global sd_result_df
    try:
        if sd_result_df is None:
            messagebox.showerror("Error", "No GSD result to save.")
            return
        save_path = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("CSV", "*.csv")])
        if save_path:
            sd_result_df.to_csv(save_path, index=False)
            messagebox.showinfo("Saved", f"GSD result saved to: {save_path}")
    except Exception as e:
        messagebox.showerror("Save Error", str(e))

tk.Button(tab2, text="Save GSD Result", command=save_gsd_result).pack(pady=5)
    
def save_steps():
    global step_result_df
    try:
        if step_result_df is None:
            messagebox.showerror("Error", "No steps to save.")
            return
        save_path = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("CSV", "*.csv")])
        if save_path:
            step_result_df.to_csv(save_path, index=False)
            messagebox.showinfo("Saved", f"Steps result saved to: {save_path}")
    except Exception as e:
        messagebox.showerror("Save Error", str(e))

tk.Button(tab2, text="Save Step Counts", command=save_steps).pack(pady=5)

# --------------------------Tab 3 Ayush's Functions----------------------------------
def load_data():
    global extracted_df
    file_path = filedialog.askopenfilename(filetypes=[("CSV/JSON files", "*.csv *.json")])
    if not file_path:
        return

    try:
        if file_path.endswith(".csv"):
            df = pd.read_csv(file_path)
        elif file_path.endswith(".json"):
            df = pd.read_json(file_path)
        else:
            raise ValueError("Unsupported file format")

        extracted_df = df.copy()
        messagebox.showinfo("Loaded", f"Loaded file:\n{file_path}")
    except Exception as e:
        messagebox.showerror("Load Error", str(e))

tk.Button(tab3, text="Upload CSV/JSON File", command=load_data).pack(pady=10)
label_frame = tk.Frame(tab3)
label_frame.pack()


# linear functions
def run_linear():
    global extracted_df
    from walking import find_walking 
    from hs_to_detection import get_hs_to_wavelet 
    from hs_to_detection import gait_parameters  

    try:
        data = pd.read_csv(extracted_df)
        data['TgF'] = np.sqrt(data['Ax']**2+data['Ay']**2+data['Az']**2)
        R_acc = data['TgF']-1
        fs = 100
        vm = np.array(R_acc)

        fs_walking = 10
        num_samples = int(np.size(vm) * fs_walking / fs)
        vm_resample = resample(vm, num_samples)
        num_elements_to_use = (vm_resample.size // fs_walking) * fs_walking
        vm_bout = vm_resample[:num_elements_to_use]

        cad = find_walking(vm_bout, 10, 0.3, (1.4, 2.3), 1.6, 2.5, 3, 20)
        wi = np.where(cad > 0, 1, 0)
        wi_resampled = np.repeat(wi, 100)

        change_indices = np.where(np.diff(wi_resampled) != 0)[0]
        walking = np.zeros((len(change_indices) // 2, 2), dtype=int)
        for i in range(0, len(change_indices), 2):
            walking[i // 2, 0] = change_indices[i] + 1
            walking[i // 2, 1] = change_indices[i + 1]

        # az = accleration['gFz']
        num_samples = int(len(accZ) * 100 / fs)
        resampled_az = resample(accZ, num_samples)

        df = pd.DataFrame(columns=['walking_index', 'steps', 'time', 'start_index', 'end_index', 'foot1_hs', 'foot2_hs',
                                   'foot1_to', 'foot2_to', 'gait_cycle_time_foot1', 'gait_cycle_time_foot2',
                                   'dst1_foot1', 'dst1_foot2', 'dst2_foot1', 'dst2_foot2', 'sst1_foot1', 'sst1_foot2',
                                   'sst2_foot1', 'sst2_foot2', 'stance_time1_foot1', 'stance_time1_foot2',
                                   'stance_time2_foot1', 'stance_time2_foot2', 'swing_time1_foot1', 'swing_time1_foot2',
                                   'swing_time2_foot1', 'swing_time2_foot2'])

        for i in range(len(walking)):
            walk_start = walking[i, 0]
            walk_end = walking[i, 1]
            walk = resampled_az[walk_start:walk_end]
            hs_idx, to_idx, invalid_hs_idx = get_hs_to_wavelet(walk, 100, 0, 1, "Test", wavelet_type_straight="gaus1", wavelet_type_turning="gaus1")
            hs_idx = hs_idx.astype(float) / 100
            to_idx = to_idx.astype(float) / 100
            df = gait_parameters(hs_idx, to_idx, df, i, walk_start, walk_end, cad, fs, 100)

        save_path = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("CSV", "*.csv")])
        if save_path:
            df.to_csv(save_path, index=False)
            messagebox.showinfo("Saved", f"Feature extraction saved to: {save_path}")
    except Exception as e:
        messagebox.showerror("Error extracting features", str(e))

def run_nonlinear():
    from ami_stergio import AMI_Stergiou
    from fnn import FNN
    from ent_Ap import Ent_Ap
    from ent_symbolic import Ent_Symbolic
    from entMSPlus import Ent_MS_Plus
    from entPermu import Ent_Permu
    from lye_R import LyE_R
    from lye_W import LyE_W
    from rqa import RQA
    from ent_Samp import Ent_Samp
    from walking import find_walking 
    from hs_to_detection import get_hs_to_wavelet 
    from hs_to_detection import gait_parameters 
    
    global extracted_df
    
    try:
        data = pd.read_csv(extracted_df)
        data['TgF'] = np.sqrt(data['Ax']**2+data['Ay']**2+data['Az']**2)
        R_acc = data['TgF']-1
        fs = 100
        vm = np.array(R_acc)
        fs_walking = 10
        num_samples = int(np.size(vm) * fs_walking / fs)
        vm_resample = resample(vm, num_samples)
        num_elements_to_use = (vm_resample.size // fs_walking) * fs_walking
        vm_bout = vm_resample[:num_elements_to_use]
        cad = find_walking(vm_bout, 10, 0.3, (1.4, 2.3), 1.6, 2.5, 3, 20)
        wi = np.where(cad > 0, 1, 0)
        wi_resampled = np.repeat(wi, 100)
        change_indices = np.where(np.diff(wi_resampled) != 0)[0]
        walking = np.zeros((len(change_indices) // 2, 2), dtype=int)
        for i in range(0, len(change_indices), 2):
            walking[i // 2, 0] = change_indices[i] + 1
            walking[i // 2, 1] = change_indices[i + 1]
        res_acc = R_acc + 1
        num_samples = int(len(res_acc) * 100 / fs)
        resampled_res_acc = resample(res_acc, num_samples)
        df2 = pd.DataFrame(columns=['walking_index','start_index', 'end_index','delay','dimension','Ap_en','Samp_en','Sym_en','Permu_en','LyEl_R','LyEs_R','Lye_W','REC','DET','MeanL','MaxL','EntrL','LAM','MeanV','MaxV','EntrV','EntrW','RCMSE','CMSE','MSE','MSE_SCALE1','MSE_SCALE2'])
        for i in range(len(walking)):
            walk_start = walking[i,0]
            walk_end= walking[i,1]
            data = resampled_res_acc[walk_start:walk_end]
            tau, V_ami = AMI_Stergiou(data,35)
            delay = int(tau[0][0])
            ans = FNN(data, delay, 30, 15, 2, 0)
            dim = ans[1]
            r = 0.2
            AE = Ent_Ap(data, dim, r)
            Samp_Ent = Ent_Samp(data, 2, r)
            Sym_En = Ent_Symbolic(data, 2)
            permEnt_value,_ = Ent_Permu(data, 2, delay)
            LyEl,LyEs,_ = LyE_R(data, "test",100,delay,2,[1,2,3,4],1,True)
            _,lye = LyE_W(data,100,delay,dim,2)
            rqa_RESULTS = RQA(data, 'RQA', dim, delay,1,'NON',1,'radius',1,0,'1',1)[0]
            REC=rqa_RESULTS['REC']
            DET=rqa_RESULTS['DET']
            MeanL=rqa_RESULTS['MeanL']
            MaxL=rqa_RESULTS['MaxL']
            EntrL=rqa_RESULTS['EntrL']
            LAM=rqa_RESULTS['LAM']
            MeanV=rqa_RESULTS['MeanL']
            MaxV=rqa_RESULTS['MaxV']
            EntrV=rqa_RESULTS['EntrV']
            EntrW=rqa_RESULTS['EntrW']
            result = Ent_MS_Plus(data,20,2,0.2)
            RCMSE = result[0].astype(float)
            CMSE = result[1].astype(float)
            MSE = np.array([(item, np.nan, np.nan) if not isinstance(item, tuple) else item for item in result[2]])
            max_len = MSE.shape[0]
            lists = [i+1,walk_start,walk_end,delay,dim,AE,Samp_Ent,Sym_En,permEnt_value,LyEl,LyEs,lye,REC,DET,MeanL,MaxL,EntrL,LAM,MeanV,MaxV,EntrV,EntrW]
            padded_lists = [np.pad(np.array([v], dtype=float), (0, max_len - 1), constant_values=np.nan) for v in lists]
            padded_lists = np.vstack(padded_lists).T
            new_row = pd.DataFrame({'walking_index':padded_lists[:,0],'start_index':padded_lists[:,1], 'end_index':padded_lists[:,2],'delay':padded_lists[:,3],'dimension':padded_lists[:,4],'Ap_en':padded_lists[:,5],'Samp_en':padded_lists[:,6],'Sym_en':padded_lists[:,7],'Permu_en':padded_lists[:,8],'LyEl_R':padded_lists[:,9],'LyEs_R':padded_lists[:,10],'Lye_W':padded_lists[:,11],'REC':padded_lists[:,12],'DET':padded_lists[:,13],'MeanL':padded_lists[:,14],'MaxL':padded_lists[:,15],'EntrL':padded_lists[:,16],'LAM':padded_lists[:,17],'MeanV':padded_lists[:,18],'MaxV':padded_lists[:,19],'EntrV':padded_lists[:,20],'EntrW':padded_lists[:,21],'RCMSE':RCMSE,'CMSE':CMSE,'MSE':MSE[:,0],'MSE_SCALE1':MSE[:,1],'MSE_SCALE2':MSE[:,2]})
            df2 = pd.concat([df2, new_row], ignore_index=True)
        save_path = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("CSV", "*.csv")])
        if save_path:
            df2.to_csv(save_path, index=False)
            messagebox.showinfo("Saved", f"Non-linear features saved to: {save_path}")
    except Exception as e:
        messagebox.showerror("Non-linear Feature Error", str(e))
        
tk.Button(tab3, text="Linear ", command=run_linear).pack(pady=20)
tk.Button(tab3, text="Non-Linear", command=run_nonlinear).pack(pady=20)


# Start the GUI loop
root.mainloop()



Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Users\ahala\AppData\Local\Programs\Python\Python312\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\ahala\AppData\Local\Temp\ipykernel_31608\556013391.py", line 303, in run_linear
    from walking import find_walking
ModuleNotFoundError: No module named 'walking'
Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Users\ahala\AppData\Local\Programs\Python\Python312\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\ahala\AppData\Local\Temp\ipykernel_31608\556013391.py", line 303, in run_linear
    from walking import find_walking
ModuleNotFoundError: No module named 'walking'
Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Users\ahala\AppData\Local\Programs\Python\Python312\Lib\tkinter\__init__.py", line 1968, in __ca

In [41]:
# !pip install ssqueezepy